In [10]:
from pathlib import Path
import os
import pandas as pd


try:
    PROJECT_DIR = Path(__file__).resolve().parents[0]
except NameError:
    PROJECT_DIR = Path(os.getcwd()).resolve().parents[0]
    
DATA_PATH = PROJECT_DIR / "outputs" / "sales_data.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

Finding Key performance indicators

In [11]:
df = pd.read_csv(DATA_PATH,parse_dates=["InvoiceDate"])
df["LineValue"] = df["Quantity"] * df["UnitPrice"]

total_revenue  = df["LineValue"].sum()
total_orders = df["InvoiceNo"].nunique()
total_customers = df["CustomerID"].nunique()
total_units = df["Quantity"].sum()
avg_order_val = (total_revenue / total_orders if total_orders else 0)

kpis = pd.DataFrame(
    [
        {
            "total_gross_sales_gbp": total_revenue,
            "total_orders": total_orders,
            "units_sold": total_units,
            "average_order_value": avg_order_val,
        }
    ]
)

kpis.to_csv(OUTPUT_DIR/"kpis.csv",index=False)
print("\n-----KEY PERFORMANCE INDICATORS-----")
print(kpis.T.to_string(header=False))

/tmp/ipykernel_2085/1931895521.py:1: DtypeWarning: Columns (0: InvoiceNo) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH,parse_dates=["InvoiceDate"])



-----KEY PERFORMANCE INDICATORS-----
total_gross_sales_gbp  1.064211e+07
total_orders           1.996200e+04
units_sold             5.572420e+06
average_order_value    5.331185e+02


Finding monthly sales

In [12]:
df["Month"] = df["InvoiceDate"].dt.to_period("M").astype(str)

monthly_sales = (
    df.groupby("Month", as_index=False).agg(
        Revenue=("LineValue", "sum"),
        Orders=("InvoiceNo", "nunique"),
        Units=("Quantity","sum"),
    ).sort_values("Month")
)

monthly_sales.to_csv(OUTPUT_DIR/"monthly_sales.csv",index=False)
print("\n----Monthly Sales----")
print(monthly_sales.to_string(index=False))


----Monthly Sales----
  Month     Revenue  Orders  Units
2010-12  821452.730    1559 358019
2011-01  689811.610    1086 387099
2011-02  522545.560    1100 282934
2011-03  716215.260    1454 376599
2011-04  536968.491    1246 307953
2011-05  769296.610    1681 395001
2011-06  760547.010    1533 388511
2011-07  718076.121    1476 399693
2011-08  757841.380    1361 421020
2011-09 1056435.192    1838 569573
2011-10 1151263.730    2040 621029
2011-11 1503866.780    2769 751377
2011-12  637790.330     819 313612


Top Monthly Sales

In [13]:
product_sales = (
    df.groupby(["StockCode","Description"], as_index=False).agg(
        Revenue=("LineValue", "sum"),
        Units=("Quantity","sum"),
        Orders=("InvoiceNo", "nunique"),
    ).sort_values("Revenue", ascending=False)
)
product_sales.to_csv(OUTPUT_DIR/"product_sales.csv")

print("\n----TOP PRODUCTS BY REVENUE----")
print(product_sales.head(10).to_string(index=False))


----TOP PRODUCTS BY REVENUE----
StockCode                        Description   Revenue  Units  Orders
      DOT                     DOTCOM POSTAGE 206248.77    706     706
    22423           REGENCY CAKESTAND 3 TIER 174156.54  13851    1988
    23843        PAPER CRAFT , LITTLE BIRDIE 168469.60  80995       1
   85123A WHITE HANGING HEART T-LIGHT HOLDER 104284.24  37580    2189
    47566                      PARTY BUNTING  99445.23  18283    1685
   85099B            JUMBO BAG RED RETROSPOT  94159.81  48371    2089
    23166     MEDIUM CERAMIC TOP STORAGE JAR  81700.92  78033     247
     POST                            POSTAGE  78101.88   3150    1126
        M                             Manual  77750.27   6984     289
    23084                 RABBIT NIGHT LIGHT  66870.03  30739     994


Country Performance

In [14]:
country_sales = (
    df.groupby("Country", as_index=False).agg(
        Revenue=("LineValue", "sum"),
        Orders=("InvoiceNo", "nunique"),
        Units=("Quantity","sum"),
    ).sort_values("Revenue",ascending=False)
)

country_sales.to_csv(OUTPUT_DIR/"country_sales.csv",index=False)
print("\n----TOP COUNTRIES BY REVENUE----")
print(country_sales.head(10).to_string(index=False))


----TOP COUNTRIES BY REVENUE----
       Country     Revenue  Orders   Units
United Kingdom 9001744.094   18020 4646906
   Netherlands  285446.340      94  200361
          EIRE  283140.520     288  147007
       Germany  228678.400     458  119154
        France  209625.370     392  112060
     Australia  138453.810      57   83891
         Spain   61558.560      90   27933
   Switzerland   57067.600      54   30617
       Belgium   41196.340      98   23237
        Sweden   38367.830      36   36078


Customer Performance

In [15]:
identified_sales = df.dropna(subset=["CustomerID"]).copy()

customer_sales = (
    identified_sales.groupby("CustomerID", as_index=False).agg(
        Revenue=("LineValue", "sum"),
                Orders=("InvoiceNo", "nunique"),
                Units=("Quantity","sum"),
    ).sort_values("Revenue", ascending=False)
)

customer_sales.to_csv(OUTPUT_DIR/"customer_sales.csv")

print("\n-----TOP IDENTIFIED CUSTOMERS-----")
print(customer_sales.head(10).to_string(index=False))


-----TOP IDENTIFIED CUSTOMERS-----
 CustomerID   Revenue  Orders  Units
    14646.0 280206.02      73 196915
    18102.0 259657.30      60  64124
    17450.0 194390.79      46  69973
    16446.0 168472.50       2  80997
    14911.0 143711.17     201  80240
    12415.0 124914.53      21  77374
    14156.0 117210.08      55  57768
    17511.0  91062.38      31  64549
    16029.0  80850.84      63  40108
    12346.0  77183.60       1  74215
